# 5 — Streaming and Online Anomaly Detection

**Sensor Intelligence Platform** — analytical walkthrough (5 / 7)

Batch analysis assumes the whole series is in memory. Production monitoring does not: readings arrive one at a time, indefinitely. `StreamProcessor` consumes a reading stream, keeps a **bounded sliding window per sensor** (constant memory — the backpressure guarantee), runs detection once per fixed-size **batch** so per-tick cost is bounded, and emits **de-duplicated** alerts to a pluggable sink.

1. Turn a simulated fleet into a reading stream.
2. Configure and run the processor.
3. Verify the bounded-memory and batching behaviour.
4. Confirm online detection recovers the same faults as a full-batch pass.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

NAVY, ORANGE, TEAL, RED, GREY = "#1f3a5f", "#e8893b", "#2a9d8f", "#c0392b", "#9aa0ad"

## 5.1 A reading stream from the fleet

We simulate three channels with injected faults, then flatten the frame into a time-ordered stream of `SensorReading`s via `iter_readings`. Sorting by timestamp interleaves the sensors exactly as a live feed would deliver them.

In [2]:
from sensor_intelligence.simulation import (
    SensorSimulator, SimulationConfig, AnomalyInjection, default_fleet,
)
from sensor_intelligence.streaming import iter_readings

PERIOD = 96
# Low daily-amplitude channels, so a point detector keys on the spike rather
# than being blinded by a strong seasonal swing (see notebook 6 for the
# deseasonalising step high-amplitude channels need).
channels = ['vibration', 'pressure', 'supply_voltage']
fleet = [s for s in default_fleet() if s.sensor_id in channels]
config = SimulationConfig(
    sensors=fleet, n_steps=PERIOD * 5, step_seconds=900, seed=23,
    anomalies=[
        AnomalyInjection('vibration', start_step=PERIOD * 2 + 40, magnitude=0.5),
        AnomalyInjection('pressure', start_step=PERIOD * 3 + 10, magnitude=9.0),
        AnomalyInjection('supply_voltage', start_step=PERIOD * 3 + 60, magnitude=13.0),
    ],
)
df = SensorSimulator(config).run().sort_values('timestamp').reset_index(drop=True)
stream = list(iter_readings(zip(df.sensor_id, df.timestamp, df.value)))
print(f'{len(stream)} readings across {df.sensor_id.nunique()} sensors')
stream[0]

1440 readings across 3 sensors


SensorReading(sensor_id='pressure', timestamp=Timestamp('2024-01-01 00:00:00'), value=101.53124651322385, unit=None, quality=1.0)

## 5.2 Configure the processor

The processor pairs any `AnomalyDetector` with an `AlertSink`. `StreamingConfig` sizes the sliding window (max readings retained **per sensor**), the warm-up `min_window` before detection starts, and the `batch_size` that triggers an inference tick. We use the robust z-score detector and collect alerts in memory.

In [3]:
from sensor_intelligence.streaming import StreamProcessor, StreamingConfig, InMemoryAlertSink
from sensor_intelligence.anomaly import RobustZScoreDetector

cfg = StreamingConfig(window_size=2 * PERIOD, min_window=PERIOD, batch_size=24)
sink = InMemoryAlertSink()
processor = StreamProcessor(
    detector=RobustZScoreDetector(window_size=PERIOD, threshold=4.0),
    sink=sink, config=cfg,
)
cfg

StreamingConfig(window_size=192, min_window=96, batch_size=24)

## 5.3 Run the stream

`run` consumes the iterator in `batch_size` chunks and returns the total alerts emitted. Because each finding is de-duplicated by timestamp, a fault that sits in the window across many ticks pages the operator **once**.

In [4]:
total = processor.run(iter(stream))
print(f'{total} alerts emitted')

table = pd.DataFrame([
    {'time': a.timestamp.strftime('%Y-%m-%d %H:%M'), 'sensor': a.sensor_id,
     'severity': a.severity.value,
     'score': round(max(an.score for an in a.anomalies), 1),
     'reason': (a.reason_codes[0] if a.reason_codes else '')}
    for a in sink.alerts
]).sort_values('time').reset_index(drop=True)
table

3 alerts emitted


,time,sensor,severity,score,reason
0,2024-01-03 10:00,vibration,critical,13.5,modified z-score +13.55 exceeds 4
1,2024-01-04 02:30,pressure,critical,7.0,modified z-score +7.01 exceeds 4
2,2024-01-04 15:00,supply_voltage,critical,6.0,modified z-score +6.01 exceeds 4


## 5.4 Bounded memory and batched inference

The per-sensor buffer is a `deque(maxlen=window_size)`, so memory is **constant** no matter how long the stream runs. We replay the stream batch by batch and record the largest per-sensor buffer after each tick (reaching into the processor's internal buffers purely to illustrate): it climbs during warm-up, then plateaus at the window cap and never grows again.

In [5]:
proc2 = StreamProcessor(
    detector=RobustZScoreDetector(window_size=PERIOD, threshold=4.0),
    sink=InMemoryAlertSink(), config=cfg,
)
sizes, batch = [], []
for r in stream:
    batch.append(r)
    if len(batch) >= cfg.batch_size:
        proc2.process_batch(batch)
        sizes.append(max(len(b) for b in proc2._buffers.values()))
        batch = []
if batch:
    proc2.process_batch(batch)
    sizes.append(max(len(b) for b in proc2._buffers.values()))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(1, len(sizes) + 1), sizes, 'o-', color=NAVY, label='largest per-sensor buffer')
ax.axhline(cfg.window_size, color=RED, ls='--', lw=1.4, label='window_size cap')
ax.set(title='Per-sensor memory is bounded by window_size',
       xlabel='inference tick (batch)', ylabel='readings buffered', ylim=(0, cfg.window_size * 1.1))
ax.legend(fontsize=9); fig.tight_layout()

## 5.5 Online detection matches the batch pass

Does processing incrementally cost accuracy? We run the same detector once over each sensor's **entire** series and compare the flagged timestamps. The streaming pass recovers the same faults — it de-duplicates, so its counts are at most the batch counts, but every fault is caught. It trades no detection quality for its bounded memory.

In [6]:
from sensor_intelligence.domain import TimeSeriesWindow

def batch_flags(frame, sid):
    s = frame[frame.sensor_id == sid].sort_values('timestamp')
    w = TimeSeriesWindow(sensor_id=sid, timestamps=list(s.timestamp),
                         values=[float(v) for v in s.value])
    return {a.timestamp for a in RobustZScoreDetector(window_size=PERIOD, threshold=4.0).detect(w)}

stream_flags = {sid: set() for sid in channels}
for a in sink.alerts:
    stream_flags[a.sensor_id].add(a.timestamp)

rows = []
for sid in channels:
    b = batch_flags(df, sid)
    s = stream_flags[sid]
    rows.append({'sensor': sid, 'batch_flags': len(b), 'streaming_alerts': len(s),
                 'in_both': len(b & s)})
pd.DataFrame(rows)

,sensor,batch_flags,streaming_alerts,in_both
0,vibration,1,1,1
1,pressure,1,1,1
2,supply_voltage,1,1,1


## Takeaways

- `StreamProcessor` runs detection over a **bounded per-sensor window**, so memory is constant for an unbounded stream.
- Inference fires **once per batch**, decoupling per-tick cost from arrival rate (backpressure safety).
- Alerts are **de-duplicated by timestamp** and pushed to a pluggable `AlertSink` (in-memory here; logging or webhook in production).
- Online detection **recovers the same faults** as a full-batch pass — see those alerts ranked into an ops report in [notebook 6](06_fleet_monitoring.ipynb).